In [2]:
# 0. SID4 parameters — run this notebook in Colab with a GPU
import time
import random
import subprocess

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

SID4 = 670
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("SID4: %04d" % SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)
print("HP_ID is reported only. HW2 has no HP_ID mapping.")
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise SystemExit("No CUDA. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")
print("GPU:", torch.cuda.get_device_name(0))


SID4: 0670
SEED: 670
SLICE: 670
HP_ID: 4
CLS_A: 0
CLS_B: 5
HP_ID is reported only. HW2 has no HP_ID mapping.
torch: 2.11.0+cu128
cuda available: True
GPU: Tesla T4


# Sneha Singh
## DATA 266 — Homework 2, Part 3 (CUDA mixed precision only)

Same 12×512 MLP, same random batch, batch 64, 40 Adam steps, lr 0.001, seed 670.
This notebook is **only** FP32 vs real CUDA fp16 AMP. Do not rerun embeddings or RAG here.


In [3]:
# GPU name and CUDA version from nvidia-smi
print(subprocess.check_output(["nvidia-smi"], text=True))


Sun Sep  6 02:06:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
# same model / data / batch / steps as optimizations.ipynb
DIM = 512
DEPTH = 12
N_CLASS = 10
BATCH = 64
STEPS = 40
LR = 0.001
REPEATS = 3

device = torch.device("cuda")
g = torch.Generator().manual_seed(SEED)
X = torch.randn(BATCH, DIM, generator=g).to(device)
y = torch.randint(0, N_CLASS, (BATCH,), generator=g).to(device)


class DeepNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.inp = nn.Linear(DIM, DIM)
        self.blocks = nn.ModuleList(
            [nn.Sequential(nn.Linear(DIM, DIM), nn.ReLU()) for _ in range(DEPTH)]
        )
        self.out = nn.Linear(DIM, N_CLASS)

    def forward(self, x):
        x = self.inp(x)
        for blk in self.blocks:
            x = blk(x)
        return self.out(x)


def train_once(model, X, y, use_amp=False):
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    last = None
    for _ in range(STEPS):
        opt.zero_grad(set_to_none=True)
        if use_amp:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                loss = loss_fn(model(X), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()
        last = loss.detach()
    return float(last)


print("X", tuple(X.shape), "on", X.device)
print("batch", BATCH, "steps", STEPS, "depth", DEPTH, "width", DIM, "lr", LR)


X (64, 512) on cuda:0
batch 64 steps 40 depth 12 width 512 lr 0.001


In [5]:
# prove autocast is fp16, warmup once, then time fp32 vs amp (mean of 3)
torch.manual_seed(SEED)
probe = DeepNet().to(device).eval()
with torch.no_grad():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        amp_out_dtype = probe(X[:2]).dtype
    fp32_out_dtype = probe(X[:2]).dtype
del probe
print("forward dtype without autocast:", fp32_out_dtype)
print("forward dtype with autocast(cuda, float16):", amp_out_dtype)
print("GradScaler: torch.amp.GradScaler('cuda') when AMP=True")
print("used_fp16:", amp_out_dtype == torch.float16)

torch.manual_seed(SEED)
warm = DeepNet().to(device)
train_once(warm, X, y, use_amp=False)
torch.cuda.synchronize()
del warm
torch.cuda.empty_cache()
print("warmup done (untimed)")

rows = []
for use_amp in [False, True]:
    times = []
    last_loss = None
    last_mem = None
    for _ in range(REPEATS):
        torch.manual_seed(SEED)
        m = DeepNet().to(device)
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        last_loss = train_once(m, X, y, use_amp=use_amp)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
        last_mem = torch.cuda.max_memory_allocated() / (1024 ** 2)
        del m
    rows.append({
        "mixed_precision": use_amp,
        "seconds_mean_of_3": round(sum(times) / len(times), 4),
        "peak_mem_mb": round(last_mem, 1),
        "final_loss": round(last_loss, 4),
        "autocast": "none (fp32)" if not use_amp else "cuda float16 + GradScaler",
        "forward_dtype": str(fp32_out_dtype) if not use_amp else str(amp_out_dtype),
    })
    print("amp", use_amp, "times", [round(t, 4) for t in times])

amp_df = pd.DataFrame(rows)
print()
print(amp_df.to_string(index=False))
print()
print("COPY THIS BLOCK BACK TO CURSOR")
print(amp_df.to_csv(index=False))
amp_df


forward dtype without autocast: torch.float32
forward dtype with autocast(cuda, float16): torch.float16
GradScaler: torch.amp.GradScaler('cuda') when AMP=True
used_fp16: True
warmup done (untimed)
amp False times [0.2315, 0.2049, 0.2666]
amp True times [0.351, 0.2303, 0.2252]

 mixed_precision  seconds_mean_of_3  peak_mem_mb  final_loss                  autocast forward_dtype
           False             0.2343         82.6      1.4924               none (fp32) torch.float32
            True             0.2688         82.6      1.4793 cuda float16 + GradScaler torch.float16

COPY THIS BLOCK BACK TO CURSOR
mixed_precision,seconds_mean_of_3,peak_mem_mb,final_loss,autocast,forward_dtype
False,0.2343,82.6,1.4924,none (fp32),torch.float32
True,0.2688,82.6,1.4793,cuda float16 + GradScaler,torch.float16



,mixed_precision,seconds_mean_of_3,peak_mem_mb,final_loss,autocast,forward_dtype
0,False,0.2343,82.6,1.4924,none (fp32),torch.float32
1,True,0.2688,82.6,1.4793,cuda float16 + GradScaler,torch.float16
